In [ ]:
import json
import os
import pandas as pd
from tqdm import tqdm

In [ ]:
dataset_names = [
    "archeology",
    "astronomy",
    "biomedical",
    "environment",
    "legal",
    "wildfire",
]

In [ ]:
for dataset_name in dataset_names:
    print(f"Processing benchmark of dataset {dataset_name}...")
    with open(f"{dataset_name}/{dataset_name}.json", "r") as f:
        data = json.load(f)
        print(f"=> Number of original questions: {len(data)}")

        valid_question_count = 0
        valid_questions = []
        for item in data:
            data_sources = item.get("data_sources", [])
            # Count questions whose data sources all are either xlsx or csv
            if all(ds.endswith(".xlsx") or ds.endswith(".csv") for ds in data_sources):
                valid_question_count += 1
                valid_questions.append(
                    {
                        "id": item["id"],
                        "query": item["query"],
                        "answer": item["answer"],
                        "answer_type": item["answer_type"],
                        "data_sources": data_sources,
                    }
                )
        valid_questions_path = f"{dataset_name}/{dataset_name}_tabular.json"
        with open(valid_questions_path, "w") as f:
            json.dump(valid_questions, f, indent=4)
        print(
            f"=> Number of questions with only xlsx/csv data sources: {valid_question_count} ({valid_question_count / len(data) * 100:.2f}%)"
        )

In [ ]:
def folder_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for name in files:
            fp = os.path.join(root, name)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total


def sizeof_fmt(num, suffix="B"):
    for unit in ["", "K", "M", "G", "T"]:
        if num < 1024:
            return f"{num:.2f}{unit}{suffix}"
        num /= 1024


for dataset_name in dataset_names:
    dataset_path = f"{dataset_name}/dataset"
    print(f"Parsing dataset: {dataset_name}")

    size_bytes = folder_size(dataset_path)
    print(f"=> Dataset size: {sizeof_fmt(size_bytes)} ({size_bytes} bytes)")

    table_count = 0
    columns_count = 0
    rows_count = 0

    for filename in tqdm(os.listdir(dataset_path)):
        if filename.endswith(".csv"):
            print(f"Parsing file: {filename}")
            table = pd.read_csv(os.path.join(dataset_path, filename))
            table_count += 1
            columns_count += len(table.columns)
            rows_count += len(table)

    print(f"=> Table count: {table_count}")
    print(f"=> Average columns per table: {columns_count / table_count}")
    print(f"=> Average rows per table: {rows_count / table_count}")
    print()